# SpaceX Falcon 9 — Launch Site Locations Analysis with Folium

In [ ]:
!pip install folium -q
import folium, pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians
spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

### Task 1 — mark all launch sites

In [ ]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
for _, r in launch_sites_df.iterrows():
    folium.Circle([r['Lat'], r['Long']], radius=1000, color='#d35400', fill=True).add_child(folium.Popup(r['Launch Site'])).add_to(site_map)
    folium.map.Marker([r['Lat'], r['Long']], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % r['Launch Site'])).add_to(site_map)
site_map

All sites are in the USA, close to the coast and as far south as possible.

### Task 2 — success/failed launches per site

In [ ]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})
site_map.add_child(marker_cluster)
for _, r in spacex_df.iterrows():
    folium.Marker([r['Lat'], r['Long']], icon=folium.Icon(color='white', icon_color=r['marker_color'])).add_to(marker_cluster)
site_map

### Task 3 — distances between a launch site and its proximities

In [ ]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
site_map.add_child(MousePosition(position='topright', separator=' Long: ', empty_string='NaN', lng_first=False,
    num_digits=20, prefix='Lat:', lat_formatter=formatter, lng_formatter=formatter))
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))
launch_site = [28.56230197, -80.57735648]  # CCAFS SLC-40
points = {'Coastline': [28.56342, -80.56794], 'Railway': [28.57206, -80.58527],
          'Highway': [28.56335, -80.57085], 'City (Melbourne)': [28.10473, -80.64531]}
for name, p in points.items():
    d = calculate_distance(*launch_site, *p)
    print(f'{name}: {d:.2f} km')
    folium.Marker(p, icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % f'{name} {d:.2f} KM')).add_to(site_map)
    folium.PolyLine(locations=[launch_site, p], weight=1).add_to(site_map)
site_map

Launch sites are close to coastline (~0.9 km), highway (~0.7 km) and railway (~1.4 km) but ~51 km from the nearest city.